In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
import os
!pip install datasets

from datasets import load_dataset

algebra_ds = load_dataset("EleutherAI/hendrycks_math", "algebra")
geometry_ds = load_dataset("EleutherAI/hendrycks_math", "geometry")
number_theory_ds = load_dataset("EleutherAI/hendrycks_math", "number_theory")
precalculus_ds = load_dataset("EleutherAI/hendrycks_math", "precalculus")

subjects = [
    "algebra",
    "counting_and_probability",
    "geometry",
    "intermediate_algebra",
    "number_theory",
    "prealgebra",
    "precalculus"
]

datasets_list = []

for subject in subjects:
    ds = load_dataset("EleutherAI/hendrycks_math", subject)
    datasets_list.append(ds["train"])

from datasets import concatenate_datasets

combined_ds = concatenate_datasets(datasets_list)
combined_ds = combined_ds.add_column("original_index", list(range(len(combined_ds))))

# Ensure the directory exists before saving and mounting drive later
# os.makedirs("/content/drive/MyDrive/math_datasets", exist_ok=True)

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

combined_ds.save_to_disk("/content/drive/MyDrive/math_datasets/hendrycks_math_full")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Mounted at /content/drive


Saving the dataset (0/1 shards):   0%|          | 0/7500 [00:00<?, ? examples/s]

In [ ]:
print(combined_ds)
print(combined_ds[0])

Dataset({
    features: ['problem', 'level', 'type', 'solution', 'original_index'],
    num_rows: 7500
})
{'problem': 'Let \\[f(x) = \\left\\{\n\\begin{array}{cl} ax+3, &\\text{ if }x>2, \\\\\nx-5 &\\text{ if } -2 \\le x \\le 2, \\\\\n2x-b &\\text{ if } x <-2.\n\\end{array}\n\\right.\\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper).', 'level': 'Level 5', 'type': 'Algebra', 'solution': 'For the piecewise function to be continuous, the cases must "meet" at $2$ and $-2$. For example, $ax+3$ and $x-5$ must be equal when $x=2$. This implies $a(2)+3=2-5$, which we solve to get $2a=-6 \\Rightarrow a=-3$. Similarly, $x-5$ and $2x-b$ must be equal when $x=-2$. Substituting, we get $-2-5=2(-2)-b$, which implies $b=3$. So $a+b=-3+3=\\boxed{0}$.', 'original_index': 0}


PREPROCESSING

In [ ]:
def preprocess(example):
    example["text"] = f"Problem:\n{example['problem']}\n\nSolution:\n{example['solution']}"
    return example

processed_ds = combined_ds.map(preprocess)

In [ ]:
# imo_ds = load_dataset("TamasSimonds/imo-dataset")

In [ ]:
# def preprocess_imo(example):
#     example["text"] = f"Problem:\n{example['prompt']}\n\nSolution:\n{example['completion']}"
#     return example

# imo_processed = imo_ds["train"].map(preprocess_imo)

In [ ]:
processed_ds.save_to_disk("/content/drive/MyDrive/math_datasets/hendrycks_processed")
# imo_processed.save_to_disk("/content/drive/MyDrive/math_datasets/imo_processed")

Saving the dataset (0/1 shards):   0%|          | 0/7500 [00:00<?, ? examples/s]

In [ ]:
split_ds = processed_ds.train_test_split(test_size=0.1)
train_ds = split_ds["train"]
test_ds = split_ds["test"]

LOAD MODEL

In [ ]:
!pip install transformers peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-Math-1.5B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/676 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [ ]:
sample = test_ds[0]
problem = sample["problem"]

prompt = f"""Solve this olympiad math problem step by step.

Problem:
{problem}

Clearly explain your reasoning and state the final answer at the end.
"""

WITHOUT FINE TUNING

In [ ]:
import re
import json
import torch
from datasets import load_from_disk
import os # Added import for os module

# =====================================================
# CREATE FIXED HENDRYCKS ALGEBRA BENCHMARK (if it doesn't exist)
# =====================================================
fixed_benchmark_path = "/content/drive/MyDrive/math_datasets/fixed_algebra_500"

if not os.path.exists(fixed_benchmark_path):
    print(f"Fixed algebra benchmark not found at {fixed_benchmark_path}. Creating it now...")
    # Load the processed dataset (assuming 'hendrycks_processed' exists from prior steps)
    loaded_processed_ds = load_from_disk("/content/drive/MyDrive/math_datasets/hendrycks_processed")

    # Filter for algebra problems
    algebra_ds_filtered = loaded_processed_ds.filter(lambda x: x["type"] == "Algebra")

    # Take the first 500 samples for a fixed benchmark
    # Ensure not to exceed available samples if less than 500
    fixed_algebra_benchmark = algebra_ds_filtered.select(range(min(500, len(algebra_ds_filtered))))

    # Save the fixed benchmark
    fixed_algebra_benchmark.save_to_disk(fixed_benchmark_path)
    print(f"Created fixed algebra benchmark with {len(fixed_algebra_benchmark)} questions.")
else:
    print(f"Fixed algebra benchmark found at {fixed_benchmark_path}. Loading existing dataset.")


# =====================================================
# LOAD FIXED HENDRYCKS ALGEBRA BENCHMARK
# =====================================================

fixed_algebra_test = load_from_disk(
    fixed_benchmark_path
)

print("Loaded fixed benchmark questions:", len(fixed_algebra_test))


# =====================================================
# ROBUST ANSWER EXTRACTION
# =====================================================

def extract_boxed_answer(text):

    # Try boxed answer first
    boxed = re.findall(r'\\boxed\{(.*?)\}', text)

    if boxed:
        return boxed[-1].strip()

    # Fallback to last number
    fallback = re.findall(r'[-+]?\d*\.?\d+', text)

    if fallback:
        return fallback[-1]

    # If no valid answer found
    return None


# =====================================================
# EVALUATION SETTINGS
# =====================================================

correct = 0
total_questions = len(fixed_algebra_test)

results = []


# =====================================================
# MAIN LOOP
# =====================================================

for i in range(total_questions):

    sample = fixed_algebra_test[i]
    problem = sample["problem"]

    prompt = f"""Solve this olympiad algebra problem step by step.

Problem:
{problem}

Clearly explain every reasoning step and state the final answer in \boxed{{}} format.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    predicted_answer = extract_boxed_answer(response)
    actual_answer = extract_boxed_answer(sample["solution"])


    # =====================================================
    # STRICT ACCURACY LOGIC
    # =====================================================

    # If predicted answer is missing, auto-fail
    if predicted_answer is None:
        is_correct = False

    # If actual answer missing, also fail
    elif actual_answer is None:
        is_correct = False

    # Normal comparison
    else:
        is_correct = predicted_answer == actual_answer


    if is_correct:
        correct += 1


    # =====================================================
    # SAVE RESULTS
    # =====================================================

    results.append({
        "question_id": i,
        "original_dataset_index": sample["original_index"],
        "category": sample["type"],
        "difficulty": sample["level"],
        "problem": sample["problem"],
        "generated_solution": response,
        "actual_solution": sample["solution"],
        "predicted_answer": predicted_answer if predicted_answer is not None else "NO ANSWER",
        "actual_answer": actual_answer if actual_answer is not None else "UNKNOWN",
        "correct": is_correct
    })


    # =====================================================
    # DISPLAY RESULTS
    # =====================================================

    print("\n" + "=" * 120)
    print(f"QUESTION {i+1}/{total_questions}")
    print(f"Original Dataset Index : {sample['original_index']}")
    print(f"Category               : {sample['type']}")
    print(f"Difficulty Level       : {sample['level']}")
    print("=" * 120)

    print("\nALGEBRA PROBLEM:\n")
    print(sample["problem"])

    print("\nMODEL STEP-BY-STEP GENERATED SOLUTION:\n")
    print(response)

    print("\nACTUAL HENDRYCKS DATASET SOLUTION:\n")
    print(sample["solution"])

    print("\nFINAL ANSWER COMPARISON:")
    print(f"Predicted Answer       : {predicted_answer if predicted_answer is not None else 'NO ANSWER'}")
    print(f"Actual Answer          : {actual_answer if actual_answer is not None else 'UNKNOWN'}")
    print(f"Correct                : {'YES' if is_correct else 'NO'}")

    print("=" * 120 + "\n")


# =====================================================
# FINAL PERFORMANCE SUMMARY
# =====================================================

accuracy = (correct / total_questions) * 100

print("\nFINAL PERFORMANCE SUMMARY")
print("=" * 70)
print(f"Dataset Used           : Hendrycks MATH (Algebra Only)")
print(f"Total Questions Tested : {total_questions}")
print(f"Correct Answers        : {correct}")
print(f"Incorrect Answers      : {total_questions - correct}")
print(f"Final Accuracy         : {accuracy:.2f}%")
print("=" * 70)


# =====================================================
# SAVE DETAILED RESULTS
# =====================================================

with open(
    "/content/drive/MyDrive/math_datasets/baseline_results_detailed.json",
    "w"
) as f:
    json.dump(results, f, indent=4)

print("Detailed results saved successfully.")

Fixed algebra benchmark not found at /content/drive/MyDrive/math_datasets/fixed_algebra_500. Creating it now...


Filter:   0%|          | 0/7500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Created fixed algebra benchmark with 500 questions.
Loaded fixed benchmark questions: 500


...


QUESTION 497/500
Original Dataset Index : 496
Category               : Algebra
Difficulty Level       : Level 2

ALGEBRA PROBLEM:

Compute $\displaystyle \frac{2+4-8+16+32-64}{4+8-16+32+64-128}$.

MODEL STEP-BY-STEP GENERATED SOLUTION:

Solve this olympiad algebra problem step by step.

Problem:
Compute $\displaystyle \frac{2+4-8+16+32-64}{4+8-16+32+64-128}$.

Clearly explain every reasoning step and state the final answer in oxed{} format.
To solve the given problem, we need to evaluate the expression:

\[
\frac{2 + 4 - 8 + 16 + 32 - 64}{4 + 8 - 16 + 32 + 64 - 128}
\]

We will break this down into two parts: the numerator and the denominator. Let's evaluate each part step by step.

1. Evaluate the numerator: \(2 + 4 - 8 + 16 + 32 - 64\)
2. Evaluate the denominator: \(4 + 8 - 16 + 32 + 64 - 128\)
3. Divide the result of the numerator by the result of the denominator.

Let's start by evaluating the numerator and the denominator separately using Python code.
```python
# Evaluate the nu

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



QUESTION 498/500
Original Dataset Index : 497
Category               : Algebra
Difficulty Level       : Level 4

ALGEBRA PROBLEM:

The area of the floor in a square room is 225 square feet. The homeowners plan to cover the floor with rows of 6-inch by 6-inch tiles. How many tiles will be in each row?

MODEL STEP-BY-STEP GENERATED SOLUTION:

Solve this olympiad algebra problem step by step.

Problem:
The area of the floor in a square room is 225 square feet. The homeowners plan to cover the floor with rows of 6-inch by 6-inch tiles. How many tiles will be in each row?

Clearly explain every reasoning step and state the final answer in oxed{} format.
To solve this problem, we need to follow these steps:

1. Determine the side length of the square room.
2. Convert the side length from feet to inches.
3. Determine the number of 6-inch tiles that fit along one side of the room.

Let's break it down:

1. The area of the floor is 225 square feet. Since the room is square, the side length \(

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



QUESTION 499/500
Original Dataset Index : 498
Category               : Algebra
Difficulty Level       : Level 4

ALGEBRA PROBLEM:

If $5^x=100$, what is the value of $5^{x+2}$?

MODEL STEP-BY-STEP GENERATED SOLUTION:

Solve this olympiad algebra problem step by step.

Problem:
If $5^x=100$, what is the value of $5^{x+2}$?

Clearly explain every reasoning step and state the final answer in oxed{} format.
To solve the problem, we need to find the value of \(5^{x+2}\) given that \(5^x = 100\).

Let's break it down step-by-step:

1. We know that \(5^x = 100\).
2. We need to find \(5^{x+2}\). Using the properties of exponents, we can rewrite \(5^{x+2}\) as \(5^x \cdot 5^2\).
3. Since \(5^x = 100\), we can substitute this value into the expression: \(5^{x+2} = 100 \cdot 5^2\).
4. We know that \(5^2 = 25\), so \(5^{x+2} = 100 \cdot 25\).
5. Therefore, \(5^{x+2} = 2500\).

Let's confirm this with Python code to ensure accuracy.
```python
# Given value
x_value = 100

# Calculate 5^(x+2)
resul

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

FINE TUNING

In [ ]:
from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(
r=16,
lora_alpha=32,
target_modules=["q_proj", "v_proj"],
lora_dropout=0.05,
bias="none",
task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


TOKENIZATION

In [ ]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [ ]:
train_tokenized = train_ds.map(tokenize_function, batched=False)
test_tokenized = test_ds.map(tokenize_function, batched=False)

Map:   0%|          | 0/6750 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

TRAIN THE MODEL

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/qwen_math_finetuned",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.268108,0.263881
2,0.279160,0.260497


TrainOutput(global_step=3376, training_loss=0.3405177485886343, metrics={'train_runtime': 6371.6641, 'train_samples_per_second': 2.119, 'train_steps_per_second': 0.53, 'total_flos': 5.443281616896e+16, 'train_loss': 0.3405177485886343, 'epoch': 2.0})

In [ ]:
trainer.save_model("/content/drive/MyDrive/qwen_math_final")
tokenizer.save_pretrained("/content/drive/MyDrive/qwen_math_final")

('/content/drive/MyDrive/qwen_math_final/tokenizer_config.json',
 '/content/drive/MyDrive/qwen_math_final/chat_template.jinja',
 '/content/drive/MyDrive/qwen_math_final/tokenizer.json')

In [ ]:
import re
import json
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import os

# =====================================================
# DEVICE SETUP
# =====================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =====================================================
# LOAD BASE MODEL + FINE-TUNED LORA MODEL
# =====================================================

model_name = "Qwen/Qwen2.5-Math-1.5B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load base quantized model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load your fine-tuned LoRA checkpoint
model = PeftModel.from_pretrained(
    base_model,
    "/content/drive/MyDrive/qwen_math_final"
)

model.eval()

# =====================================================
# CREATE FIXED HENDRYCKS ALGEBRA BENCHMARK (if not exist)
# =====================================================

fixed_benchmark_path = "/content/drive/MyDrive/math_datasets/fixed_algebra_500"

if not os.path.exists(fixed_benchmark_path):
    print(f"Fixed algebra benchmark not found at {fixed_benchmark_path}. Creating it now...")

    loaded_processed_ds = load_from_disk(
        "/content/drive/MyDrive/math_datasets/hendrycks_processed"
    )

    algebra_ds_filtered = loaded_processed_ds.filter(
        lambda x: x["type"] == "Algebra"
    )

    fixed_algebra_benchmark = algebra_ds_filtered.select(
        range(min(500, len(algebra_ds_filtered)))
    )

    fixed_algebra_benchmark.save_to_disk(fixed_benchmark_path)

    print(
        f"Created fixed algebra benchmark with {len(fixed_algebra_benchmark)} questions."
    )

else:
    print(
        f"Fixed algebra benchmark found at {fixed_benchmark_path}. Loading existing dataset."
    )

# =====================================================
# LOAD FIXED BENCHMARK
# =====================================================

fixed_algebra_test = load_from_disk(
    fixed_benchmark_path
)

print("Loaded fixed benchmark questions:", len(fixed_algebra_test))

# =====================================================
# ROBUST ANSWER EXTRACTION
# =====================================================

def extract_boxed_answer(text):

    boxed = re.findall(r'\\boxed\{(.*?)\}', text)

    if boxed:
        return boxed[-1].strip()

    fallback = re.findall(r'[-+]?\d*\.?\d+', text)

    if fallback:
        return fallback[-1]

    return None


# =====================================================
# EVALUATION SETTINGS
# =====================================================

correct = 0
total_questions = len(fixed_algebra_test)

results = []

# =====================================================
# MAIN LOOP
# =====================================================

for i in range(total_questions):

    sample = fixed_algebra_test[i]
    problem = sample["problem"]

    prompt = f"""Solve this olympiad algebra problem step by step.

Problem:
{problem}

Clearly explain every reasoning step and state the final answer in \\boxed{{}} format.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    predicted_answer = extract_boxed_answer(response)
    actual_answer = extract_boxed_answer(sample["solution"])

    # =====================================================
    # STRICT ACCURACY LOGIC
    # =====================================================

    if predicted_answer is None:
        is_correct = False
    elif actual_answer is None:
        is_correct = False
    else:
        is_correct = predicted_answer == actual_answer

    if is_correct:
        correct += 1

    # =====================================================
    # SAVE RESULTS
    # =====================================================

    results.append({
        "question_id": i,
        "original_dataset_index": sample["original_index"],
        "category": sample["type"],
        "difficulty": sample["level"],
        "problem": sample["problem"],
        "generated_solution": response,
        "actual_solution": sample["solution"],
        "predicted_answer": predicted_answer if predicted_answer is not None else "NO ANSWER",
        "actual_answer": actual_answer if actual_answer is not None else "UNKNOWN",
        "correct": is_correct
    })

    # =====================================================
    # DISPLAY RESULTS
    # =====================================================

    print("\n" + "=" * 120)
    print(f"QUESTION {i+1}/{total_questions}")
    print(f"Original Dataset Index : {sample['original_index']}")
    print(f"Category               : {sample['type']}")
    print(f"Difficulty Level       : {sample['level']}")
    print("=" * 120)

    print("\nALGEBRA PROBLEM:\n")
    print(sample["problem"])

    print("\nFINE-TUNED MODEL STEP-BY-STEP GENERATED SOLUTION:\n")
    print(response)

    print("\nACTUAL HENDRYCKS DATASET SOLUTION:\n")
    print(sample["solution"])

    print("\nFINAL ANSWER COMPARISON:")
    print(
        f"Predicted Answer       : {predicted_answer if predicted_answer is not None else 'NO ANSWER'}"
    )
    print(
        f"Actual Answer          : {actual_answer if actual_answer is not None else 'UNKNOWN'}"
    )
    print(f"Correct                : {'YES' if is_correct else 'NO'}")

    print("=" * 120 + "\n")

# =====================================================
# FINAL PERFORMANCE SUMMARY
# =====================================================

accuracy = (correct / total_questions) * 100

print("\nFINAL PERFORMANCE SUMMARY")
print("=" * 70)
print(f"Dataset Used           : Hendrycks MATH (Algebra Only)")
print(f"Model Type             : Fine-Tuned Qwen2.5-Math + LoRA")
print(f"Total Questions Tested : {total_questions}")
print(f"Correct Answers        : {correct}")
print(f"Incorrect Answers      : {total_questions - correct}")
print(f"Final Accuracy         : {accuracy:.2f}%")
print("=" * 70)

# =====================================================
# SAVE DETAILED RESULTS
# =====================================================

with open(
    "/content/drive/MyDrive/math_datasets/fine_tuned_results_detailed.json",
    "w"
) as f:
    json.dump(results, f, indent=4)

print("Fine-tuned detailed results saved successfully.")

Using device: cuda


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Fixed algebra benchmark found at /content/drive/MyDrive/math_datasets/fixed_algebra_500. Loading existing dataset.
Loaded fixed benchmark questions: 500


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...


QUESTION 497/500
Original Dataset Index : 496
Category               : Algebra
Difficulty Level       : Level 2

ALGEBRA PROBLEM:

Compute $\displaystyle \frac{2+4-8+16+32-64}{4+8-16+32+64-128}$.

FINE-TUNED MODEL STEP-BY-STEP GENERATED SOLUTION:

Solve this olympiad algebra problem step by step.

Problem:
Compute $\displaystyle \frac{2+4-8+16+32-64}{4+8-16+32+64-128}$.

Clearly explain every reasoning step and state the final answer in \boxed{} format.
To solve the given problem, we need to compute the value of the expression \(\frac{2+4-8+16+32-64}{4+8-16+32+64-128}\).

Let's break it down step by step:

1. Compute the numerator \(2 + 4 - 8 + 16 + 32 - 64\).
2. Compute the denominator \(4 + 8 - 16 + 32 + 64 - 128\).
3. Divide the result of the numerator by the result of the denominator.

Let's start by calculating the numerator and the denominator separately.

### Step 1: Calculate the numerator
The numerator is \(2 + 4 - 8 + 16 + 32 - 64\).

### Step 2: Calculate the denominator
Th

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



QUESTION 498/500
Original Dataset Index : 497
Category               : Algebra
Difficulty Level       : Level 4

ALGEBRA PROBLEM:

The area of the floor in a square room is 225 square feet. The homeowners plan to cover the floor with rows of 6-inch by 6-inch tiles. How many tiles will be in each row?

FINE-TUNED MODEL STEP-BY-STEP GENERATED SOLUTION:

Solve this olympiad algebra problem step by step.

Problem:
The area of the floor in a square room is 225 square feet. The homeowners plan to cover the floor with rows of 6-inch by 6-inch tiles. How many tiles will be in each row?

Clearly explain every reasoning step and state the final answer in \boxed{} format.
To solve this problem, we need to follow these steps:

1. Determine the side length of the square room.
2. Convert the side length from feet to inches.
3. Determine the number of 6-inch tiles that fit along one side of the room.

Let's break it down:

1. The area of the floor is 225 square feet. Since the room is square, the si

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



QUESTION 499/500
Original Dataset Index : 498
Category               : Algebra
Difficulty Level       : Level 4

ALGEBRA PROBLEM:

If $5^x=100$, what is the value of $5^{x+2}$?

FINE-TUNED MODEL STEP-BY-STEP GENERATED SOLUTION:

Solve this olympiad algebra problem step by step.

Problem:
If $5^x=100$, what is the value of $5^{x+2}$?

Clearly explain every reasoning step and state the final answer in \boxed{} format.
To solve the problem, we need to find the value of \(5^{x+2}\) given that \(5^x = 100\).

Let's break it down step-by-step:

1. We know that \(5^x = 100\).
2. We need to find \(5^{x+2}\). Using the properties of exponents, we can rewrite \(5^{x+2}\) as \(5^x \cdot 5^2\).
3. Since \(5^x = 100\), we can substitute this value into the expression: \(5^{x+2} = 100 \cdot 5^2\).
4. We know that \(5^2 = 25\), so \(5^{x+2} = 100 \cdot 25\).
5. Therefore, \(5^{x+2} = 2500\).

Let's confirm this with Python code.
```python
# Given that 5^x = 100
# We need to find the value of 5^(x+2)